# PeekaBook 멀티세션 시뮬레이션 — Colab Pro Plus

**환경:** Colab Pro Plus + T4 GPU + 공유 드라이브  
**예상 실행 시간:** 5페르소나 × 30세션, 동시 2개 → 약 6시간

## 사전 준비

1. ✅ **Colab Pro Plus 결제**
2. ✅ **공유 드라이브 단축키 추가**: drive.google.com → 공유 문서함 → `peekabook_public_drive` 우클릭 → "드라이브에 바로가기 추가"
3. ✅ **Runtime 설정**: 상단 메뉴 → Runtime → Change runtime type → **T4 GPU**
4. ✅ **Colab Secrets 설정**: 왼쪽 🔑 아이콘 → 다음 키 추가:
   - `OPENAI_API_KEY`
   - `ANTHROPIC_API_KEY`
   - `WANDB_API_KEY`
   - `QDRANT_URL`
   - `QDRANT_API_KEY`
   - `QDRANT_COLLECTION_NAME` (예: `books_intro_48k`)
   - `GITHUB_TOKEN` (private repo clone용)
   - 각 Secret 마다 "Notebook access" 토글 켜기

## 1. Drive 마운트 + 경로 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 공유 폴더 보이는지 확인
!ls /content/drive/MyDrive/peekabook_public_drive/

In [ ]:
# 결과 저장 경로 확정
OUTPUT_DIR = "/content/drive/MyDrive/peekabook_public_drive/simulation_results"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
!ls "$OUTPUT_DIR"
print(f"\n✓ OUTPUT_DIR: {OUTPUT_DIR}")

## 2. GitHub Repo 가져오기

Private repo이므로 GITHUB_TOKEN 사용. 이미 clone되어 있으면 `git pull`만.

In [ ]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_OWNER   = "aiffelpeekabook"
REPO_NAME    = "AIFFEL_final_project_peekabook"
REPO_PATH    = f"/content/{REPO_NAME}"
REPO_URL     = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
else:
    %cd {REPO_PATH}
    !git pull
    %cd /content

print(f"\n✓ REPO_PATH: {REPO_PATH}")

## 3. 의존성 설치

requirements.txt 우선 설치, 누락된 게 있으면 개별 설치.

In [ ]:
%cd {REPO_PATH}
!pip install -q -r requirements.txt

## 4. 환경 변수 설정

Colab Secrets에서 로드해 os.environ에 주입.  
`run_multi_simulation.py`가 `.env` 파일도 읽지만, Secrets가 우선.

In [ ]:
import os
from google.colab import userdata

required = [
    'OPENAI_API_KEY',
    'ANTHROPIC_API_KEY',
    'WANDB_API_KEY',
    'QDRANT_URL',
    'QDRANT_API_KEY',
    'QDRANT_COLLECTION_NAME',
]

missing = []
for key in required:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    print(f"⚠️  누락된 Secrets: {missing}")
    print(f"   왼쪽 🔑 아이콘에서 추가하세요.")
else:
    print(f"✓ 환경 변수 {len(required)}개 설정 완료")
    print(f"  collection: {os.environ['QDRANT_COLLECTION_NAME']}")

## 5. GPU 확인

T4 GPU가 할당됐는지 확인. Reranker가 이 GPU를 사용.

In [ ]:
!nvidia-smi

import torch
print(f"\nPyTorch CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"디바이스: {torch.cuda.get_device_name(0)}")
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 6. 시뮬레이션 설정

**페르소나 선택 방식 (하나만 활성화):**
- `PERSONAS = "A_최재원,B_한미영"` — 콤마 구분 복수
- `PERSONAS = "A_최재원"` — 단일
- `PERSONA_SLICE = "0:3"` — 인덱스 슬라이싱 (전체에서 0~2번)
- `USE_ALL = True` — 전체

**Resume:** 런타임 끊긴 후 재실행 시 `RUN_ID`에 기존 ID 입력.  
이미 완료된 페르소나는 자동으로 skip.

In [ ]:
# ── 페르소나 선택 (하나만 활성화) ────────────────────────
PERSONAS      = "A_최재원,B_한미영"   # 콤마 구분 복수
# PERSONAS      = "A_최재원"            # 단일
PERSONA_SLICE = None                  # 예: "0:3"
USE_ALL       = False                 # True 시 전체 페르소나

# ── 시뮬레이션 파라미터 ─────────────────────────────────
N_SESSIONS = 30
CONCURRENT = 2          # T4에서 권장
QUERY_TRANSFORM   = "none"
USE_GENRE_FILTER  = False

# ── Resume (이전 런타임이 끊겼을 때만 채우기) ────────────
RUN_ID = None
# RUN_ID = "20260619_103000"  # 예시: 기존 run 이어서 진행

# ── 경로 ────────────────────────────────────────────────
PERSONA_DIR = f"{REPO_PATH}/backend/data/personas"

print(f"  PERSONAS:        {PERSONAS or PERSONA_SLICE or ('ALL' if USE_ALL else None)}")
print(f"  N_SESSIONS:      {N_SESSIONS}")
print(f"  CONCURRENT:      {CONCURRENT}")
print(f"  QUERY_TRANSFORM: {QUERY_TRANSFORM}")
print(f"  RUN_ID:          {RUN_ID or '(새로 생성)'}")
print(f"  OUTPUT_DIR:      {OUTPUT_DIR}")
print(f"  PERSONA_DIR:     {PERSONA_DIR}")

## 7. 시뮬레이션 실행

결과는 `OUTPUT_DIR/{run_id}/{persona_id}.json` 으로 페르소나마다 저장.  
런타임이 끊겨도 이미 완료된 페르소나는 보존됨.

In [ ]:
%cd {REPO_PATH}

# 페르소나 선택 인자 결정
if USE_ALL:
    persona_arg = "--all"
elif PERSONAS:
    persona_arg = f"--personas '{PERSONAS}'"
elif PERSONA_SLICE:
    persona_arg = f"--persona-slice '{PERSONA_SLICE}'"
else:
    raise ValueError("PERSONAS, PERSONA_SLICE, USE_ALL 중 하나는 설정해야 함")

# Resume 인자
resume_arg = f"--run-id {RUN_ID}" if RUN_ID else ""

# 장르 필터 인자
genre_arg = "--use-genre-filter" if USE_GENRE_FILTER else ""

cmd = f"""python research/tests/lael/run_multi_simulation.py \\
    {persona_arg} \\
    --persona-dir {PERSONA_DIR} \\
    --n-sessions {N_SESSIONS} \\
    --concurrent {CONCURRENT} \\
    --query-transform {QUERY_TRANSFORM} \\
    --output-dir '{OUTPUT_DIR}' \\
    {genre_arg} {resume_arg}"""

print(f"실행 명령:\n{cmd}\n")
print("=" * 60)

!{cmd}

## 8. 결과 확인

공유 드라이브에 저장된 최신 run의 결과 파일 목록.

In [ ]:
import os, json, glob

runs = sorted(glob.glob(f"{OUTPUT_DIR}/*/"))
if not runs:
    print("저장된 run이 없습니다.")
else:
    latest = runs[-1].rstrip("/")
    print(f"최신 run: {os.path.basename(latest)}")
    print(f"경로: {latest}\n")

    files = sorted(glob.glob(f"{latest}/*.json"))
    persona_files = [f for f in files if not os.path.basename(f).startswith("_")]
    print(f"페르소나 결과 {len(persona_files)}개:")
    for f in persona_files:
        size_kb = os.path.getsize(f) / 1024
        with open(f) as fp:
            data = json.load(fp)
        n_sessions = len(data.get("sessions", []))
        print(f"  ✓ {os.path.basename(f):30s} | sessions: {n_sessions:2d} | {size_kb:.1f} KB")

## 9. (선택) 런타임 종료

시뮬레이션 끝나면 컴퓨트 유닛 절약을 위해 런타임 종료.

In [ ]:
# from google.colab import runtime
# runtime.unassign()